In [ ]:
!which python

In [1]:
from pathlib import Path

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "src" / "data" / "SmartEM" / "Philips").exists():
    repo_root = repo_root.parents[2]

data_dir = repo_root / "src" / "data" / "SmartEM" / "Philips"

input_data = pd.read_csv(data_dir / "input.csv")
output_data = pd.read_csv(data_dir / "output.csv")

# input_data, output_data

In [2]:
# input_data.nunique()
# Drop columns with only one unique value
input_data = input_data.loc[:, input_data.nunique() > 1]
# input_data.nunique()
# Number of rows in input_data and output_data
# len(input_data), len(output_data)

In [13]:
from sklearn.model_selection import train_test_split # Function to split data into training and testing sets
def trainTestSplit(input_data, output_data):
    train_inputs = []
    test_inputs = []
    train_outputs = []
    test_outputs = []

    unique_configs = sorted(input_data["input_1"].unique()) #Disticnt values in input_1 column

    for config in unique_configs:
        mask = input_data["input_1"] == config

        X = input_data[mask]
        Y = output_data[mask]

        X_train, X_test, Y_train, Y_test = train_test_split(
            X, Y, test_size=0.2, random_state=42
        )

        train_inputs.append(X_train)
        test_inputs.append(X_test)
        train_outputs.append(Y_train)
        test_outputs.append(Y_test)

    return train_inputs, test_inputs, train_outputs, test_outputs

from xgboost import XGBRegressor

def train_xgboost(X_train, y_train):
    model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    enable_categorical=True,
    verbosity=2
)

    model.fit(X_train, y_train, verbose=True)
    return model

from lightgbm import LGBMRegressor
def train_lightgbm(X_train, y_train):
    model = LGBMRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)
    return model

from catboost import CatBoostRegressor
def train_catboost(X_train, y_train, categorical_columns=None):
    model = CatBoostRegressor(
        iterations=300,
        depth=6,
        learning_rate=0.05,
        random_seed=42,
        verbose=10
    )

    model.fit(
        X_train,
        y_train,
        cat_features=categorical_columns
    )

    return model

In [14]:

Xtrain, Xtest, Ytrain, Ytest = trainTestSplit(input_data, output_data)

In [ ]:
X_train = Xtrain[0]
y_train = Ytrain[0]["output_1"]

categorical_columns = [
    "input_2", "input_3", "input_4", "input_5",
    "input_6", "input_7", "input_8", "input_9",
    "input_10", "input_11"
]

X_train = Xtrain[0].copy().drop(columns=["input_1"])
X_test  = Xtest[0].copy().drop(columns=["input_1"])

for col in categorical_columns:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype("category")
        X_test[col] = X_test[col].astype("category")

print(X_train.dtypes)

print("Training XGBoost...")
xgb_model = train_xgboost(X_train, y_train)
print("Training LightGBM...")
lgb_model = train_lightgbm(X_train, y_train)
print("Training CatBoost...")
categorical_columns = [
    "input_2", "input_3", "input_4", "input_5",
    "input_6", "input_7", "input_8", "input_9",
    "input_10", "input_11"
]
cat_model = train_catboost(X_train, y_train, categorical_columns)

input_2     category
input_3     category
input_4     category
input_5     category
input_6     category
input_7     category
input_8     category
input_9     category
input_10    category
input_11    category
input_14     float64
input_18     float64
input_19     float64
input_20     float64
input_21     float64
input_22     float64
dtype: object
Training XGBoost...
[19:54:31] INFO: /__w/xgboost/xgboost/src/data/iterative_dmatrix.cc:56: Finished constructing the `IterativeDMatrix`: (2589, 16, 41424).
Training LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000445 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1106
[LightGBM] [Info] Number of data points in the train set: 2589, number of used features: 6
[LightGBM] [Info] Start training from score 0.416377
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain

In [20]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def test_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    rmse = mse ** 0.5
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("Test results")
    print("------------")
    print(f"MSE  : {mse:.6f}")
    print(f"RMSE : {rmse:.6f}")
    print(f"MAE  : {mae:.6f}")
    print(f"R²   : {r2:.6f}")

    return y_pred
y_test = Ytest[0]["output_1"]
y_pred1 = test_model(xgb_model, X_test, y_test)
y_pred2 = test_model(lgb_model, X_test, y_test)
y_pred3 = test_model(cat_model, X_test, y_test)


Test results
------------
MSE  : 0.090381
RMSE : 0.300634
MAE  : 0.199391
R²   : 0.634208
Test results
------------
MSE  : 0.085176
RMSE : 0.291850
MAE  : 0.192399
R²   : 0.655272
Test results
------------
MSE  : 0.081750
RMSE : 0.285919
MAE  : 0.193470
R²   : 0.669140


In [ ]:
# Train all models then create an ensemble method to predict the output. Compare the ensemble method's performance to the individual models.